# K-Means Clustering - Allrecipes.com

### Question:
How effectively can K-Means clustering be applied to segment recipes, based on macronutrient composition and preparation time, in order to support time-constrained individuals in making nutritionally informed choices?

#### Overview:
To answer the question above, the [all_recipes.csv](https://github.com/owlzyseyes/tastyR/tree/main/data-raw) data, which was originally scraped from [www.allrecipes.com](https://www.allrecipes.com/) and provided by [owlzeyes - Brian Mubia](https://github.com/owlzyseyes), has been taken from the [tastyR package](https://github.com/owlzyseyes/tastyR) (see also [tastyR package](https://cran.r-project.org/package=tastyR)).

The original [all_recipes.csv]('https://raw.githubusercontent.com/owlzyseyes/tastyR/refs/heads/main/data-raw/allrecipes.csv')[all_recipes.csv] file contains 14,426 rows and 16 columns of data. This data was chosen, rather than the smaller more curated [cuisines.csv](https://raw.githubusercontent.com/owlzyseyes/tastyR/refs/heads/main/data-raw/cuisines.csv) file, due to the requirement for demonstrating data transformation and cleansing skills. The smaller file contains 2,218 rows and 17 columns of data, the additional column is a character column 'country - The country/region the cuisine is from' and won't be used in this analysis.

*Write an overview/summary of the steps taken in here when it's finished.*

## 1. Libraries and Data Loading

In [28]:
import warnings
warnings.filterwarnings("ignore") # suppress non-critical warnings throughout the script.

# Import today's date to use in the work.
from datetime import date

In [29]:
# Standard library imports
import importlib
import subprocess
import sys

# List required packages
required_packages = [
    "pandas",          # DataFrames, joins, light transforms
    "numpy",           # Numerical ops
    "matplotlib",      # Plotting
    "seaborn",         # Statistical plots
    "sklearn",         # ML
    "statsmodels",     # Statistical modelling
    "duckdb",          # Fast SQL engine
    "pyarrow",         # Parquet / Arrow
    "ydata_profiling", # EDA (optional)
    "ydata-sdk",       # Advanced data profiling, validation, and synthetic data
]

"""
The ensure_packages function below is set up to check whether each package in the above list is installed.
- If installed it prints a confirmation message with a tick.
- If missing it installs the package using pip install. 
"""
def ensure_packages(packages):
    for pkg in packages:
        try:
            importlib.import_module(pkg)
            print(f"✓ {pkg} already installed")
        except ImportError:
            print(f"✗ {pkg} missing — installing...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# Run the package check.
ensure_packages(required_packages)

✓ pandas already installed
✓ numpy already installed
✓ matplotlib already installed
✓ seaborn already installed
✓ sklearn already installed
✓ statsmodels already installed
✓ duckdb already installed
✓ pyarrow already installed
✓ ydata_profiling already installed
✗ ydata-sdk missing — installing...


In [30]:
# Import the libraries required for this work
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [31]:
all_recipes = pd.read_csv('https://raw.githubusercontent.com/owlzyseyes/tastyR/refs/heads/main/data-raw/allrecipes.csv')
all_recipes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14426 entries, 0 to 14425
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            14426 non-null  object 
 1   url             14426 non-null  object 
 2   author          14426 non-null  object 
 3   date_published  14426 non-null  object 
 4   ingredients     14417 non-null  object 
 5   calories        14226 non-null  float64
 6   fat             14070 non-null  float64
 7   carbs           14212 non-null  float64
 8   protein         14178 non-null  float64
 9   avg_rating      13454 non-null  float64
 10  total_ratings   13454 non-null  float64
 11  reviews         13353 non-null  float64
 12  prep_time       14426 non-null  int64  
 13  cook_time       14426 non-null  int64  
 14  total_time      14426 non-null  int64  
 15  servings        14405 non-null  float64
dtypes: float64(8), int64(3), object(5)
memory usage: 1.8+ MB


In [32]:
all_recipes.head(10)

,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,Chewy Whole Wheat Peanut Butter Brownies,https://www.allrecipes.com/recipe/140717/chewy...,DMOMMY,2020-06-18,"⅓ cup margarine, softened, ⅔ cup white sugar, ...",222.0,13.0,24.0,6.0,4.4,47.0,36.0,20,35,55,16.0
1,Pumpkin Pie Eggnog,https://www.allrecipes.com/recipe/269204/pumpk...,Bobbie Susan,2022-09-26,"12 egg yolks, 2 cups heavy whipping cream, ½ ...",477.0,31.0,43.0,8.0,5.0,1.0,1.0,10,5,495,8.0
2,Eggs Poached in Tomato Sauce,https://www.allrecipes.com/recipe/238054/eggs-...,Bren,2018-06-08,"2 tablespoons olive oil, or to taste, ½ onion...",354.0,18.0,32.0,20.0,4.8,4.0,4.0,10,75,85,4.0
3,Minestrone Casserole,https://www.allrecipes.com/minestrone-casserol...,Sarah Brekke,2025-03-03,4 cups dried mafalda pasta (mini lasagna noodl...,356.0,9.0,53.0,19.0,4.3,14.0,13.0,20,40,60,8.0
4,Yummy Stuffed Peppers,https://www.allrecipes.com/recipe/241937/yummy...,Procrastigirl,2024-12-11,"4 green bell peppers, halved lengthwise and se...",366.0,22.0,23.0,19.0,4.7,84.0,67.0,30,95,125,8.0
5,Prime Rib Our Way,https://www.allrecipes.com/recipe/219586/prime...,Cajun Girl,2019-04-03,"1 (8 pound) standing rib roast, fat trimmed, 2...",709.0,47.0,31.0,37.0,4.2,5.0,3.0,45,80,155,12.0
6,Parmesan Chicken Legs,https://www.allrecipes.com/recipe/8636/parmesa...,Anika,2023-01-04,"1 large egg, 2 cups grated Parmesan cheese, 1 ...",466.0,27.0,1.0,52.0,4.4,648.0,468.0,15,45,60,6.0
7,Chicken Andouille Gumbo,https://www.allrecipes.com/recipe/12950/chicke...,Bob Cody,2022-07-14,"12 cups water, 3 pounds chicken parts, 2 table...",782.0,61.0,19.0,40.0,4.6,347.0,259.0,10,190,200,8.0
8,Sweet Pork for Burritos,https://www.allrecipes.com/recipe/185816/sweet...,Dean,2023-01-19,"3 pounds pork shoulder roast, 2 cups salsa, 1 ...",355.0,15.0,33.0,23.0,4.7,129.0,102.0,30,480,510,12.0
9,Quick Baked Chicken Parmesan,https://www.allrecipes.com/recipe/239507/quick...,ChristineM,2024-11-14,"canola oil cooking spray, ½ cup water, 1 egg,...",395.0,12.0,33.0,37.0,4.7,195.0,153.0,20,55,75,6.0


Next Steps:
1. Explore the data, deciding which columns are required.
2. Cleanse the required columns dealing with (missing values etc etc)
3. Move to k-means clustering.
4. Write report.